# Rust quad kernel: saved example parity

Run this notebook from the `demos` folder after building the extension with `maturin develop` in the project virtual environment. It compares the Rust mesh against the saved `V` and `Q` arrays in every `../data/*.json` fixture. Vertex coordinates use a tight floating-point tolerance; quad indices and ordering are exact.

Use `plot_meshes_html(sample_n=1)` to export all examples into one interactive Plotly HTML view. Its slider switches between samples.


In [ ]:
import glob
import json
import os
import numpy as np
import floorplan_backend

DATA = '../data'
assert os.path.isdir(DATA), f'Run from the demos folder; missing {DATA}'

In [3]:
def check_example(path, l_tar=30.0):
    with open(path, encoding='utf-8') as f:
        example = json.load(f)
    actual_v, actual_q = floorplan_backend.mesh_domain(example['coords'], l_tar=l_tar)
    expected_v = np.asarray(example['V'], dtype=np.float64)
    expected_q = np.asarray(example['Q'], dtype=np.int64)
    actual_v = np.asarray(actual_v, dtype=np.float64)
    actual_q = np.asarray(actual_q, dtype=np.int64)
    same_shape = actual_v.shape == expected_v.shape and actual_q.shape == expected_q.shape
    v_match = same_shape and np.allclose(actual_v, expected_v, rtol=1e-11, atol=1e-9)
    q_match = same_shape and np.array_equal(actual_q, expected_q)
    return {
        'example': os.path.basename(path),
        'V_match': bool(v_match),
        'Q_match': bool(q_match),
        'actual_vertices': len(actual_v),
        'expected_vertices': len(expected_v),
        'actual_quads': len(actual_q),
        'expected_quads': len(expected_q),
        'max_vertex_error': float(np.max(np.abs(actual_v - expected_v))) if actual_v.shape == expected_v.shape else np.nan,
    }


In [ ]:
import plotly.graph_objects as go

def plot_meshes_html(output_path=None, l_tar=30.0, sample_n=1):
    """Export every saved polygon mesh into one standalone, slider-driven HTML.

    `sample_n` is the one-based sample shown when the file opens.
    Returns the absolute path to the written HTML file.
    """
    examples = sorted(glob.glob(os.path.join(DATA, '*.json')))
    if not examples:
        raise FileNotFoundError(f'No JSON examples found in {DATA}')
    if not 1 <= sample_n <= len(examples):
        raise IndexError(f'sample_n must be between 1 and {len(examples)}')

    samples = []
    for path in examples:
        with open(path, encoding='utf-8') as f:
            payload = json.load(f)
        coords = np.asarray(payload['coords'], dtype=float)
        vertices, quads = floorplan_backend.mesh_domain(payload['coords'], l_tar=l_tar)
        vertices = np.asarray(vertices, dtype=float)

        # One line trace per mesh keeps the Plotly figure compact and responsive.
        edge_x, edge_y = [], []
        for quad in quads:
            loop = list(quad) + [quad[0]]
            edge_x.extend([vertices[i, 0] for i in loop] + [None])
            edge_y.extend([vertices[i, 1] for i in loop] + [None])
        boundary = np.vstack([coords, coords[0]])
        extent = np.vstack([vertices, coords])
        lo = extent.min(axis=0)
        hi = extent.max(axis=0)
        center = 0.5 * (lo + hi)
        span = max(float((hi - lo).max()), 1.0)
        half = 0.55 * span  # 10% padding around the larger dimension
        axis_range = [[float(center[0] - half), float(center[0] + half)],
                      [float(center[1] - half), float(center[1] + half)]]
        samples.append({
            'name': os.path.basename(path),
            'title': f'{os.path.splitext(os.path.basename(path))[0]}: {len(vertices)} vertices, {len(quads)} quads',
            'edge_x': edge_x, 'edge_y': edge_y,
            'boundary_x': boundary[:, 0].tolist(),
            'boundary_y': boundary[:, 1].tolist(),
            'axis_range': axis_range,
        })

    selected = samples[sample_n - 1]
    fig = go.Figure(data=[
        go.Scatter(x=selected['edge_x'], y=selected['edge_y'], mode='lines',
                   name='Quad mesh', line=dict(color='#1769aa', width=1), hoverinfo='skip'),
        go.Scatter(x=selected['boundary_x'], y=selected['boundary_y'], mode='lines',
                   name='Domain boundary', line=dict(color='#d62728', width=3)),
    ])
    frames = []
    for i, sample in enumerate(samples):
        frames.append(go.Frame(
            name=str(i),
            data=[
                go.Scatter(x=sample['edge_x'], y=sample['edge_y']),
                go.Scatter(x=sample['boundary_x'], y=sample['boundary_y']),
            ],
            traces=[0, 1],
            layout=go.Layout(
                title_text=sample['title'],
                xaxis=dict(range=sample['axis_range'][0]),
                yaxis=dict(range=sample['axis_range'][1]),
            ),
        ))
    fig.frames = frames
    fig.update_layout(
        title=selected['title'], template='plotly_white', hovermode='closest',
        xaxis=dict(title='X', range=selected['axis_range'][0], scaleanchor='y', scaleratio=1),
        yaxis=dict(title='Y', range=selected['axis_range'][1]),
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='left', x=0),
        sliders=[{
            'active': sample_n - 1, 'currentvalue': {'prefix': 'Sample: '},
            'pad': {'t': 45},
            'steps': [{
                'label': str(i + 1),
                'method': 'animate',
                'args': [[str(i)], {'mode': 'immediate', 'frame': {'duration': 0, 'redraw': True},
                                      'transition': {'duration': 0}}],
            } for i in range(len(samples))],
        }],
    )

    if output_path is None:
        output_path = '../mesh_html/all_examples_meshes.html'
    os.makedirs(os.path.dirname(os.path.abspath(output_path)), exist_ok=True)
    fig.write_html(output_path, include_plotlyjs=True, full_html=True)
    return os.path.abspath(output_path)


In [5]:
# Export all examples; SAMPLE_N chooses which sample is visible when the HTML opens.
SAMPLE_N = 1
html_path = plot_meshes_html(sample_n=SAMPLE_N)
print(f'Created slider visualization for {len(glob.glob(os.path.join(DATA, "*.json")))} samples.')
print(f'Initial sample: {SAMPLE_N}')
print(f'HTML: {html_path}')

Created slider visualization for 49 samples.
Initial sample: 1
HTML: C:\Users\jairl\Documents\GitHub\FloorplanGenerator\Backend\mesh_html\all_examples_meshes.html
